In [ ]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
import urllib

load_dotenv()  # đọc file .env

server = os.getenv('DB_SERVER')
database = os.getenv('DB_DATABASE')
driver = os.getenv('DB_DRIVER')

conn_str = urllib.parse.quote_plus(
    f'DRIVER={{{driver}}};SERVER={server};DATABASE={database};Trusted_Connection=yes;'
)
engine = create_engine(f'mssql+pyodbc:///?odbc_connect={conn_str}')

# --- Thứ tự load theo FK dependency ---
load_order = [
    ('olist_customers_dataset.csv', 'olist_customers'),
    ('olist_sellers_dataset.csv', 'olist_sellers'),
    ('olist_geolocation_dataset.csv', 'olist_geolocation'),
    ('product_category_name_translation.csv', 'product_category'),
    ('olist_products_dataset.csv', 'olist_products'),
    ('olist_orders_dataset.csv', 'olist_orders'),
    ('olist_order_items_dataset.csv', 'olist_order_items'),
    ('olist_order_payments_dataset.csv', 'olist_order_payments'),
]

for file_name, table_name in load_order:
    path = rf'D:\olist-customer-satisfaction-analysis\data\raw\{file_name}'
    print(f'Đang xử lý: {table_name}...')
    
    try:
        chunk_size = 5000 # Tranh crash khi load file lớn
        total_rows = 0
        for chunk in pd.read_csv(path, chunksize=chunk_size):
            chunk.to_sql(table_name, con=engine, schema='staging', if_exists='append', index=False)
            total_rows += len(chunk)
            del chunk
            gc.collect()
            
        print(f'✅ {table_name}: {total_rows} dòng đã nạp thành công.')
        
    except Exception as e:
        print(f'❌ {table_name}: LỖI — {e}')
    gc.collect() 
print("Quá trình Import kết thúc!")

Đang xử lý: olist_customers...
✅ olist_customers: 99441 dòng đã nạp thành công.
Đang xử lý: olist_sellers...
✅ olist_sellers: 3095 dòng đã nạp thành công.
Đang xử lý: olist_geolocation...
✅ olist_geolocation: 1000163 dòng đã nạp thành công.
Đang xử lý: product_category...
✅ product_category: 71 dòng đã nạp thành công.
Đang xử lý: olist_products...
✅ olist_products: 32951 dòng đã nạp thành công.
Đang xử lý: olist_orders...
✅ olist_orders: 99441 dòng đã nạp thành công.
Đang xử lý: olist_order_items...
✅ olist_order_items: 112650 dòng đã nạp thành công.
Đang xử lý: olist_order_payments...
✅ olist_order_payments: 103886 dòng đã nạp thành công.
Đang xử lý: olist_order_reviews...
✅ olist_order_reviews: 99224 dòng đã nạp thành công.
Quá trình Import kết thúc!


In [5]:
df_reviews = pd.read_csv(
    r'D:\olist-customer-satisfaction-analysis\data\raw\olist_order_reviews_dataset.csv',
    engine='python',
    on_bad_lines='warn',
    encoding='utf-8'
)
print(f"Đọc được: {len(df_reviews)} dòng")

df_reviews.to_sql('olist_order_reviews', con=engine, schema='staging', if_exists='replace', index=False)

Đọc được: 99224 dòng


255